In [0]:
%sql
-- Context
USE CATALOG assignment1;
USE SCHEMA silver;

In [0]:
%sql
-- ------------------------------------------------------------------
-- 0) Monitoring sink (one-time)
-- ------------------------------------------------------------------
CREATE SCHEMA IF NOT EXISTS assignment1.monitoring;

CREATE TABLE IF NOT EXISTS assignment1.monitoring.validation_results (
  check_time   TIMESTAMP,
  check_group  STRING,
  check_name   STRING,
  result       STRING,     -- PASS/FAIL
  expected     BIGINT,
  actual       BIGINT,
  details      STRING
) USING DELTA;

In [0]:
%sql


-- ------------------------------------------------------------------
-- 1) ROW COUNT parity: bronze vs silver
-- ------------------------------------------------------------------

-- region
INSERT INTO assignment1.monitoring.validation_results
SELECT current_timestamp(), 'rowcount', 'region rows',
       CASE WHEN s.cnt = b.cnt THEN 'PASS' ELSE 'FAIL' END,
       b.cnt, s.cnt, 'silver.region vs bronze.region'
FROM (SELECT COUNT(*) cnt FROM assignment1.bronze.region) b,
     (SELECT COUNT(*) cnt FROM assignment1.silver.region) s;

-- nation
INSERT INTO assignment1.monitoring.validation_results
SELECT current_timestamp(), 'rowcount', 'nation rows',
       CASE WHEN s.cnt = b.cnt THEN 'PASS' ELSE 'FAIL' END,
       b.cnt, s.cnt, 'silver.nation vs bronze.nation'
FROM (SELECT COUNT(*) cnt FROM assignment1.bronze.nation) b,
     (SELECT COUNT(*) cnt FROM assignment1.silver.nation) s;

-- supplier
INSERT INTO assignment1.monitoring.validation_results
SELECT current_timestamp(), 'rowcount', 'supplier rows',
       CASE WHEN s.cnt = b.cnt THEN 'PASS' ELSE 'FAIL' END,
       b.cnt, s.cnt, 'silver.supplier vs bronze.supplier'
FROM (SELECT COUNT(*) cnt FROM assignment1.bronze.supplier) b,
     (SELECT COUNT(*) cnt FROM assignment1.silver.supplier) s;

-- customer
INSERT INTO assignment1.monitoring.validation_results
SELECT current_timestamp(), 'rowcount', 'customer rows',
       CASE WHEN s.cnt = b.cnt THEN 'PASS' ELSE 'FAIL' END,
       b.cnt, s.cnt, 'silver.customer vs bronze.customer'
FROM (SELECT COUNT(*) cnt FROM assignment1.bronze.customer) b,
     (SELECT COUNT(*) cnt FROM assignment1.silver.customer) s;

-- part
INSERT INTO assignment1.monitoring.validation_results
SELECT current_timestamp(), 'rowcount', 'part rows',
       CASE WHEN s.cnt = b.cnt THEN 'PASS' ELSE 'FAIL' END,
       b.cnt, s.cnt, 'silver.part vs bronze.part'
FROM (SELECT COUNT(*) cnt FROM assignment1.bronze.part) b,
     (SELECT COUNT(*) cnt FROM assignment1.silver.part) s;

-- partsupp
INSERT INTO assignment1.monitoring.validation_results
SELECT current_timestamp(), 'rowcount', 'partsupp rows',
       CASE WHEN s.cnt = b.cnt THEN 'PASS' ELSE 'FAIL' END,
       b.cnt, s.cnt, 'silver.partsupp vs bronze.partsupp'
FROM (SELECT COUNT(*) cnt FROM assignment1.bronze.partsupp) b,
     (SELECT COUNT(*) cnt FROM assignment1.silver.partsupp) s;

-- orders
INSERT INTO assignment1.monitoring.validation_results
SELECT current_timestamp(), 'rowcount', 'orders rows',
       CASE WHEN s.cnt = b.cnt THEN 'PASS' ELSE 'FAIL' END,
       b.cnt, s.cnt, 'silver.orders vs bronze.orders'
FROM (SELECT COUNT(*) cnt FROM assignment1.bronze.orders) b,
     (SELECT COUNT(*) cnt FROM assignment1.silver.orders) s;

-- lineitem
INSERT INTO assignment1.monitoring.validation_results
SELECT current_timestamp(), 'rowcount', 'lineitem rows',
       CASE WHEN s.cnt = b.cnt THEN 'PASS' ELSE 'FAIL' END,
       b.cnt, s.cnt, 'silver.lineitem vs bronze.lineitem'
FROM (SELECT COUNT(*) cnt FROM assignment1.bronze.lineitem) b,
     (SELECT COUNT(*) cnt FROM assignment1.silver.lineitem) s;

In [0]:
%sql
-- REGION PK
INSERT INTO assignment1.monitoring.validation_results
SELECT current_timestamp(), 'primary_key', 'region pk unique',
       CASE WHEN s.cnt = s.distinct_cnt THEN 'PASS' ELSE 'FAIL' END AS result,
       CAST(s.cnt AS BIGINT)           AS expected,
       CAST(s.distinct_cnt AS BIGINT)  AS actual,
       'region.region_key unique'
FROM (
  SELECT COUNT(*) cnt, COUNT(DISTINCT region_key) distinct_cnt
  FROM assignment1.silver.region
) s;

-- NATION PK
INSERT INTO assignment1.monitoring.validation_results
SELECT current_timestamp(), 'primary_key', 'nation pk unique',
       CASE WHEN s.cnt = s.distinct_cnt THEN 'PASS' ELSE 'FAIL' END,
       CAST(s.cnt AS BIGINT), CAST(s.distinct_cnt AS BIGINT),
       'nation.nation_key unique'
FROM (SELECT COUNT(*) cnt, COUNT(DISTINCT nation_key) distinct_cnt
      FROM assignment1.silver.nation) s;

-- SUPPLIER PK
INSERT INTO assignment1.monitoring.validation_results
SELECT current_timestamp(), 'primary_key', 'supplier pk unique',
       CASE WHEN s.cnt = s.distinct_cnt THEN 'PASS' ELSE 'FAIL' END,
       CAST(s.cnt AS BIGINT), CAST(s.distinct_cnt AS BIGINT),
       'supplier.supplier_key unique'
FROM (SELECT COUNT(*) cnt, COUNT(DISTINCT supplier_key) distinct_cnt
      FROM assignment1.silver.supplier) s;

-- CUSTOMER PK
INSERT INTO assignment1.monitoring.validation_results
SELECT current_timestamp(), 'primary_key', 'customer pk unique',
       CASE WHEN s.cnt = s.distinct_cnt THEN 'PASS' ELSE 'FAIL' END,
       CAST(s.cnt AS BIGINT), CAST(s.distinct_cnt AS BIGINT),
       'customer.customer_key unique'
FROM (SELECT COUNT(*) cnt, COUNT(DISTINCT customer_key) distinct_cnt
      FROM assignment1.silver.customer) s;

-- PART PK
INSERT INTO assignment1.monitoring.validation_results
SELECT current_timestamp(), 'primary_key', 'part pk unique',
       CASE WHEN s.cnt = s.distinct_cnt THEN 'PASS' ELSE 'FAIL' END,
       CAST(s.cnt AS BIGINT), CAST(s.distinct_cnt AS BIGINT),
       'part.part_key unique'
FROM (SELECT COUNT(*) cnt, COUNT(DISTINCT part_key) distinct_cnt
      FROM assignment1.silver.part) s;

-- PARTSUPP composite PK
INSERT INTO assignment1.monitoring.validation_results
SELECT current_timestamp(), 'primary_key', 'partsupp pk unique',
       CASE WHEN s.cnt = s.distinct_cnt THEN 'PASS' ELSE 'FAIL' END,
       CAST(s.cnt AS BIGINT), CAST(s.distinct_cnt AS BIGINT),
       'partsupp (part_key, supplier_key) unique'
FROM (
  SELECT COUNT(*) cnt,
         COUNT(DISTINCT named_struct('p', part_key, 's', supplier_key)) distinct_cnt
  FROM assignment1.silver.partsupp
) s;

-- ORDERS PK
INSERT INTO assignment1.monitoring.validation_results
SELECT current_timestamp(), 'primary_key', 'orders pk unique',
       CASE WHEN s.cnt = s.distinct_cnt THEN 'PASS' ELSE 'FAIL' END,
       CAST(s.cnt AS BIGINT), CAST(s.distinct_cnt AS BIGINT),
       'orders.order_key unique'
FROM (SELECT COUNT(*) cnt, COUNT(DISTINCT order_key) distinct_cnt
      FROM assignment1.silver.orders) s;

-- LINEITEM composite PK
INSERT INTO assignment1.monitoring.validation_results
SELECT current_timestamp(), 'primary_key', 'lineitem pk unique',
       CASE WHEN s.cnt = s.distinct_cnt THEN 'PASS' ELSE 'FAIL' END,
       CAST(s.cnt AS BIGINT), CAST(s.distinct_cnt AS BIGINT),
       'lineitem (order_key, line_number) unique'
FROM (
  SELECT COUNT(*) cnt,
         COUNT(DISTINCT named_struct('o', order_key, 'l', line_number)) distinct_cnt
  FROM assignment1.silver.lineitem
) s;


In [0]:
%sql
-- nation -> region
INSERT INTO assignment1.monitoring.validation_results
SELECT current_timestamp(), 'foreign_key', 'nation.region_key -> region',
       CASE WHEN v.violations = 0 THEN 'PASS' ELSE 'FAIL' END AS result,
       CAST(0 AS BIGINT)                         AS expected,
       CAST(v.violations AS BIGINT)              AS actual,
       'nation.region_key must exist in region.region_key'
FROM (
  SELECT COUNT(*) AS violations
  FROM assignment1.silver.nation n
  LEFT ANTI JOIN assignment1.silver.region r
    ON n.region_key = r.region_key
) v;

-- customer -> nation
INSERT INTO assignment1.monitoring.validation_results
SELECT current_timestamp(), 'foreign_key', 'customer.nation_key -> nation',
       CASE WHEN v.violations = 0 THEN 'PASS' ELSE 'FAIL' END,
       CAST(0 AS BIGINT),
       CAST(v.violations AS BIGINT),
       'customer.nation_key must exist in nation.nation_key'
FROM (
  SELECT COUNT(*) AS violations
  FROM assignment1.silver.customer c
  LEFT ANTI JOIN assignment1.silver.nation n
    ON c.nation_key = n.nation_key
) v;

-- supplier -> nation
INSERT INTO assignment1.monitoring.validation_results
SELECT current_timestamp(), 'foreign_key', 'supplier.nation_key -> nation',
       CASE WHEN v.violations = 0 THEN 'PASS' ELSE 'FAIL' END,
       CAST(0 AS BIGINT),
       CAST(v.violations AS BIGINT),
       'supplier.nation_key must exist in nation.nation_key'
FROM (
  SELECT COUNT(*) AS violations
  FROM assignment1.silver.supplier s
  LEFT ANTI JOIN assignment1.silver.nation n
    ON s.nation_key = n.nation_key
) v;

-- partsupp -> part
INSERT INTO assignment1.monitoring.validation_results
SELECT current_timestamp(), 'foreign_key', 'partsupp.part_key -> part',
       CASE WHEN v.violations = 0 THEN 'PASS' ELSE 'FAIL' END,
       CAST(0 AS BIGINT),
       CAST(v.violations AS BIGINT),
       'partsupp.part_key must exist in part.part_key'
FROM (
  SELECT COUNT(*) AS violations
  FROM assignment1.silver.partsupp ps
  LEFT ANTI JOIN assignment1.silver.part p
    ON ps.part_key = p.part_key
) v;

-- partsupp -> supplier
INSERT INTO assignment1.monitoring.validation_results
SELECT current_timestamp(), 'foreign_key', 'partsupp.supplier_key -> supplier',
       CASE WHEN v.violations = 0 THEN 'PASS' ELSE 'FAIL' END,
       CAST(0 AS BIGINT),
       CAST(v.violations AS BIGINT),
       'partsupp.supplier_key must exist in supplier.supplier_key'
FROM (
  SELECT COUNT(*) AS violations
  FROM assignment1.silver.partsupp ps
  LEFT ANTI JOIN assignment1.silver.supplier s
    ON ps.supplier_key = s.supplier_key
) v;

-- orders -> customer
INSERT INTO assignment1.monitoring.validation_results
SELECT current_timestamp(), 'foreign_key', 'orders.customer_key -> customer',
       CASE WHEN v.violations = 0 THEN 'PASS' ELSE 'FAIL' END,
       CAST(0 AS BIGINT),
       CAST(v.violations AS BIGINT),
       'orders.customer_key must exist in customer.customer_key'
FROM (
  SELECT COUNT(*) AS violations
  FROM assignment1.silver.orders o
  LEFT ANTI JOIN assignment1.silver.customer c
    ON o.customer_key = c.customer_key
) v;

-- lineitem -> orders
INSERT INTO assignment1.monitoring.validation_results
SELECT current_timestamp(), 'foreign_key', 'lineitem.order_key -> orders',
       CASE WHEN v.violations = 0 THEN 'PASS' ELSE 'FAIL' END,
       CAST(0 AS BIGINT),
       CAST(v.violations AS BIGINT),
       'lineitem.order_key must exist in orders.order_key'
FROM (
  SELECT COUNT(*) AS violations
  FROM assignment1.silver.lineitem l
  LEFT ANTI JOIN assignment1.silver.orders o
    ON l.order_key = o.order_key
) v;

-- lineitem -> part
INSERT INTO assignment1.monitoring.validation_results
SELECT current_timestamp(), 'foreign_key', 'lineitem.part_key -> part',
       CASE WHEN v.violations = 0 THEN 'PASS' ELSE 'FAIL' END,
       CAST(0 AS BIGINT),
       CAST(v.violations AS BIGINT),
       'lineitem.part_key must exist in part.part_key'
FROM (
  SELECT COUNT(*) AS violations
  FROM assignment1.silver.lineitem l
  LEFT ANTI JOIN assignment1.silver.part p
    ON l.part_key = p.part_key
) v;

-- lineitem -> supplier
INSERT INTO assignment1.monitoring.validation_results
SELECT current_timestamp(), 'foreign_key', 'lineitem.supplier_key -> supplier',
       CASE WHEN v.violations = 0 THEN 'PASS' ELSE 'FAIL' END,
       CAST(0 AS BIGINT),
       CAST(v.violations AS BIGINT),
       'lineitem.supplier_key must exist in supplier.supplier_key'
FROM (
  SELECT COUNT(*) AS violations
  FROM assignment1.silver.lineitem l
  LEFT ANTI JOIN assignment1.silver.supplier s
    ON l.supplier_key = s.supplier_key
) v;

-- lineitem -> partsupp (composite)
INSERT INTO assignment1.monitoring.validation_results
SELECT current_timestamp(), 'foreign_key', 'lineitem (part,supplier) -> partsupp',
       CASE WHEN v.violations = 0 THEN 'PASS' ELSE 'FAIL' END,
       CAST(0 AS BIGINT),
       CAST(v.violations AS BIGINT),
       'lineitem.(part_key,supplier_key) must exist in partsupp'
FROM (
  SELECT COUNT(*) AS violations
  FROM assignment1.silver.lineitem l
  LEFT ANTI JOIN assignment1.silver.partsupp ps
    ON l.part_key = ps.part_key AND l.supplier_key = ps.supplier_key
) v;
